# Lughaat-TTS-82M — training session

Runs the next units of work (data prep → Stage 1 → Stage 2 → export → upload) and stops before the 12 h limit. **Re-run until it reports `done`.**

One-time setup (about 10 minutes):
1. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**, Persistence off.
2. **Add-ons → Secrets**: add `HF_TOKEN` = your Hugging Face token with *write* scope, and attach it to this notebook.
3. Accept the terms of the gated dataset at https://huggingface.co/datasets/ai4bharat/Rasa while logged in with the same account.
4. Attach the project code: either add the Kaggle Dataset `lughaat-tts-code` (a zip/upload of this repository) as an input, or set `PROJECT_GIT_URL` below.

Everything else (repos, data, checkpoints, state) is created in your Hugging Face account: `<user>/lughaat-tts-work` (private, working state) and `<user>/lughaat-tts-82m` (the model).

In [ ]:
# ===================== configuration (edit if you like) =====================
HF_USERNAME = ""            # optional: HF account/org that owns the repos (default = owner of HF_TOKEN)
PROJECT_GIT_URL = ""        # optional: git URL of this project instead of the lughaat-tts-code dataset
SESSION_BUDGET_MIN = 675     # stop cleanly after this many minutes (Kaggle limit is 12 h)
URDU_HOURS_PER_SPEAKER = 6   # Rasa hours per speaker (neutral styles first); raise for more data / longer training
ENGLISH_REPLAY_HOURS = 3     # LibriTTS-R hours mixed in so English sounds are not forgotten (never below 3)
STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 5
SHARD_MINUTES = 60           # one shard = one checkpoint + backup; a killed session loses at most this much
REPO_PRIVATE = 1             # 1 = private HF repos (flip to public later in the HF UI)
USE_INDICVOICES_R = 0        # 1 = also add IndicVoices-R Urdu (needs its gated terms accepted)
LLM_LEXICON_BUDGET_USD = 0   # >0 and an ANTHROPIC_API_KEY secret = LLM pronunciation candidates for the lexicon
FORCE_PHASE = ""             # advanced: jump to a phase, e.g. "stage2"
# ============================================================================
import os, json, shutil, subprocess, sys, glob
for k, v in dict(HF_USERNAME=HF_USERNAME, SESSION_BUDGET_MIN=SESSION_BUDGET_MIN, URDU_HOURS_PER_SPEAKER=URDU_HOURS_PER_SPEAKER,
                 ENGLISH_REPLAY_HOURS=ENGLISH_REPLAY_HOURS, STAGE1_EPOCHS=STAGE1_EPOCHS, STAGE2_EPOCHS=STAGE2_EPOCHS,
                 SHARD_MINUTES=SHARD_MINUTES, REPO_PRIVATE=REPO_PRIVATE, USE_INDICVOICES_R=USE_INDICVOICES_R,
                 LLM_LEXICON_BUDGET_USD=LLM_LEXICON_BUDGET_USD, FORCE_PHASE=FORCE_PHASE).items():
    if v not in ("", None):
        os.environ[k] = str(v)
# secrets -> environment (HF_TOKEN, optional ANTHROPIC_API_KEY)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for name in ("HF_TOKEN", "ANTHROPIC_API_KEY", "HF_USERNAME"):
        try:
            val = _s.get_secret(name)
            if val:
                os.environ[name] = val
        except Exception:
            pass
except Exception as e:
    print('kaggle_secrets unavailable:', e)
assert os.environ.get('HF_TOKEN'), 'Add the HF_TOKEN secret (Add-ons -> Secrets) and attach it to this notebook.'
print('HF_TOKEN present; GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
# ===================== get the project code =====================
PROJECT = '/kaggle/working/project'
if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT)
if PROJECT_GIT_URL:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', PROJECT_GIT_URL, PROJECT], check=True)
else:
    cands = [d for d in glob.glob('/kaggle/input/*') if os.path.exists(os.path.join(d, 'kaggle', 'train_session.py'))]
    if not cands:  # zipped upload or nested folder
        for z in glob.glob('/kaggle/input/*/*.zip'):
            shutil.unpack_archive(z, '/kaggle/working/unzipped')
        cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/working/unzipped/**/kaggle/train_session.py', recursive=True)]
        cands += [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/*/**/kaggle/train_session.py', recursive=True)]
    assert cands, 'Project code not found: attach the lughaat-tts-code dataset or set PROJECT_GIT_URL'
    shutil.copytree(cands[0], PROJECT)
os.environ['PROJECT_DIR'] = PROJECT
print('project at', PROJECT, '->', sorted(os.listdir(PROJECT)))
# dependencies (torch, transformers, accelerate come with the Kaggle image)
subprocess.run('apt-get -qq install -y espeak-ng libespeak-ng1 libsndfile1 >/dev/null 2>&1 || true', shell=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(PROJECT, 'requirements-kaggle.txt')], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=False)

In [ ]:
# ===================== run the session =====================
# Idempotent: downloads state.json from <user>/lughaat-tts-work, runs the next shards/phases, backs up after every
# shard, and exits before the Kaggle limit. Re-run this notebook until the summary says done=True.
rc = subprocess.call([sys.executable, '-u', os.path.join(PROJECT, 'kaggle', 'train_session.py')], cwd=PROJECT)
print('exit code', rc)
print(open('/kaggle/working/summary.txt').read() if os.path.exists('/kaggle/working/summary.txt') else 'no summary')
if os.path.exists('/kaggle/working/state.json'):
    st = json.load(open('/kaggle/working/state.json'))
    print(json.dumps({k: st.get(k) for k in ('phase', 'done', 'best_s1_epoch', 'best_s2_epoch', 'stop_reason', 'last_error')}, indent=1, default=str))